# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their fields, referencing by @id
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs['@id']} | name: {rs.get('name', '<no name>')}")
    if 'field' in rs:
        # field could be single dict or a list
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("   Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"      @id: {f.get('@id', f)} | name: {f.get('name', '<no name>')}")
            else:
                print(f"      @id: {f}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into Pandas DataFrames
# For this dataset, we use all record sets listed above
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Display columns from the first record set (if available)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming numeric fields, and grouping by key attributes using referenced `@id` fields.

In [ ]:
# For EDA, let's choose a numeric field available in the main record set.
# Review the first DataFrame's columns and pick a likely numeric column, such as 'cr:field/peptide_count'.

main_df = dataframes[main_record_set_id]
# Change the field name below if needed based on print above
numeric_field_id = None
for col in main_df.columns:
    # Heuristics: columns with 'count', 'MW', or 'abundance' are likely numeric
    if any(word in col.lower() for word in ['count', 'mw', 'abundance', 'coverage', 'peptide', 'score', 'intensity']):
        numeric_field_id = col
        break
if not numeric_field_id:
    print("No obvious numeric field found. Please check the available columns.")
else:
    # Drop missing/NaN and ensure float type
    edf = main_df.copy()
    edf = edf[pd.to_numeric(edf[numeric_field_id], errors='coerce').notnull()]
    edf[numeric_field_id] = edf[numeric_field_id].astype(float)
    threshold = edf[numeric_field_id].mean()  # Use mean as dynamic threshold
    filtered_df = edf[edf[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (e.g., 'cr:field/description', 'cr:field/accession')
    group_field = None
    for col in main_df.columns:
        if any(word in col.lower() for word in ['group', 'type', 'class', 'description', 'accession']):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field} (@id):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].astype(float), bins=30, kde=True)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if group_field exists
    if group_field:
        # Avoid too many categories by using the top N
        top_groups = main_df[group_field].value_counts().index[:10]
        subdf = main_df[main_df[group_field].isin(top_groups)]
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field_id, data=subdf)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} grouped by {group_field}")
        plt.show()

## 6. Conclusion
This notebook has demonstrated exploring a mass spectrometry dataset described by a Croissant schema. Using `mlcroissant`, we loaded metadata, examined record sets and fields by `@id`, extracted data into DataFrames, and performed simple EDA including filtering and normalization.

Key findings and next steps:
- The dataset contains rich tabular records describing protein identifications and abundance in human cells.
- Numeric fields such as peptide counts or abundances can be analyzed and visualized.
- The Croissant schema's `@id` referencing ensures precise linking to metadata for reproducible analyses.

You can extend this notebook by:
- Exploring additional record sets or fields via their `@id`.
- Merging related tables on shared identifiers.
- Applying domain-specific statistical or machine learning analyses.